# Hansen Ch.15 Multivariate Time Series

**Chapter 15 Multivariate Time Series**

理论推导与**面向初学者的详细注释**见同目录 `Hansen_Ch15_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：平稳性核对 + 实证 VAR/SVAR + 末尾的 **理论结论蒙特卡洛验证**。

> **写给只学过李子奈/陈强的同学：** VAR = "一元 AR 的矩阵化"——$Y_t$ 是向量，$\alpha_j$ 变成矩阵 $A_j$，平稳条件从 $|\alpha|<1$ 变成**伴随矩阵特征值**$|\lambda|<1$，脉冲响应从 $b_j=\alpha^j$ 变成 $\Theta_h=A^h$。两个新概念：
> - **正交化 IRF（OIRF）需要识别**：Cholesky $LL'=\Sigma$ 把相关残差变独立，但 $L$ **依赖变量排序**（已 MC 验证：不等方差时排序不同 OIRF 明显不同）⇒ 需经济理论的 SVAR 约束。
> - **Granger 因果** = 系数约束：$Y_2$ 不 Granger 因果 $Y_1$ ⇔ $Y_1$ 方程中 $Y_2$ 滞后系数全 0（Wald 检验）。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv, eigvals
from scipy.linalg import cholesky
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

def max_abs_eig(A):
    return float(np.max(np.abs(eigvals(A))))

# 15.1-15.2
print("15.1a", max_abs_eig([[0.7,0.2],[0.2,0.7]]))
print("15.1b", max_abs_eig([[0.8,0.4],[0.4,0.8]]))
print("15.1c", max_abs_eig([[0.8,0.4],[-0.4,0.8]]))
A1=np.array([[0.3,0.2],[0.2,0.3]]); A2=np.array([[0.4,-0.1],[-0.1,0.4]])
C=np.block([[A1,A2],[np.eye(2),np.zeros((2,2))]])
print("15.2 companion", max_abs_eig(C))

# 15.12
rho=0.8
Sig=np.array([[1.,rho],[rho,1.]])
L=cholesky(Sig, lower=True)
print("15.12 L\n", L)
Theta=np.array([[1.,0.],[1.,1.]])
print("OIRF\n", Theta@L)


In [ ]:

def make_lags(Y, p):
    T, m = Y.shape
    y = Y[p:]
    parts = [np.ones((T-p,1))]
    for j in range(1,p+1):
        parts.append(Y[p-j:T-j])
    return y, np.hstack(parts)

def estimate_var(Y, p):
    y, X = make_lags(Y, p)
    B = inv(X.T@X)@(X.T@y)
    E = y - X@B
    return B, E, E.T@E/len(y), len(y)

def companion_irf(B, p, m, hmax, shock):
    As = [B[1+(j-1)*m:1+j*m,:].T for j in range(1,p+1)]
    k=m*p
    C=np.zeros((k,k))
    C[0:m,0:m]=As[0]
    for j in range(1,p):
        C[0:m, j*m:(j+1)*m]=As[j]
    if p>1: C[m:,0:m*(p-1)]=np.eye(m*(p-1))
    irfs=[]; state=np.zeros(k); state[:m]=shock
    for h in range(hmax+1):
        irfs.append(state[:m].copy()); state=C@state
    return np.array(irfs)

def aic_var(Y,p):
    y,X=make_lags(Y,p)
    B=inv(X.T@X)@(X.T@y)
    E=y-X@B
    n,m=y.shape
    Sig=E.T@E/n
    return np.log(np.linalg.det(Sig))+2*(m*m*p+m)/n


## 15.14–15.16, 15.19–15.20 实证摘要

In [ ]:

qd=pd.read_excel(ROOT/"FRED-QD/FRED-QD.xlsx")
md=pd.read_excel(ROOT/"FRED-MD/FRED-MD.xlsx")
# 15.14
g=100*np.log(pd.to_numeric(qd["gdpc1"],errors="coerce")).diff()
pi=100*np.log(pd.to_numeric(qd["gdpctpi"],errors="coerce")).diff()
ff=pd.to_numeric(qd["fedfunds"],errors="coerce")
Y=pd.DataFrame({"g":g,"pi":pi,"ff":ff}).dropna().values
B,E,Sig,n=estimate_var(Y,6)
L=cholesky(Sig,lower=True)
irf=companion_irf(B,6,3,12,L[:,0])
print("15.14 n",n,"cum GDP supply",np.cumsum(irf[:,0])[[0,4,8,12]])

# Kilian
kil=pd.read_excel(ROOT/"Kilian2009/Kilian2009.xlsx")
Yk=kil[["oil","output","price"]].astype(float).values
Yk[:,0]*=-1
B,E,Sig,n=estimate_var(Yk,4)
L=cholesky(Sig,lower=True)
print("15.15 Kilian n",n)
for s in range(3):
    irf=companion_irf(B,4,3,12,L[:,s])
    print(" output IRF shock",s, np.round(irf[[0,6,12],1],3))

# housing
for c in ["permit","houst","realln"]:
    md[c]=pd.to_numeric(md[c],errors="coerce")
df=pd.DataFrame({"permit":md.permit,"houst":md.houst,"gloan":100*np.log(md.realln).diff()}).dropna()
aics=[(p,aic_var(df.values,p)) for p in range(1,9)]
print("15.16 AIC best", min(aics,key=lambda x:x[1]))

# 15.19 Granger
gdp=100*np.log(pd.to_numeric(qd["gdpc1"],errors="coerce")).diff()
m1n=pd.to_numeric(qd["m1realx"],errors="coerce")*pd.to_numeric(qd["cpiaucsl"],errors="coerce")
mg=100*np.log(m1n).diff()
d=pd.DataFrame({"g":gdp,"m":mg}).dropna()
g,m=d.g.values,d.m.values
p=4
Y=g[p:]
X=np.column_stack([g[p-j:len(g)-j] for j in range(1,5)]+[m[p-j:len(m)-j] for j in range(1,5)]+[np.ones(len(Y))])
b=inv(X.T@X)@(X.T@Y)
e=Y-X@b
V=inv(X.T@X)@((X*e[:,None]).T@(X*e[:,None]))@inv(X.T@X)
R=np.zeros((4,9))
for i in range(4): R[i,4+i]=1
W=float((R@b)@inv(R@V@R.T)@(R@b))
print("15.19 Granger money W",W,"p",1-stats.chi2.cdf(W,4))
print("sum money", b[4:8].sum())


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch15 的核心结论：(1) VAR(1) MA 表示 $\Theta_h=A^h$（用自协方差反推核对）；(2) VAR(2) 伴随矩阵平稳性；(3) Cholesky 正交 IRF 的**排序敏感性**（不等方差下两种排序 OIRF 明显不同）；(4) Granger 因果的 Wald 检验。可独立运行。

In [ ]:
import numpy as np
from scipy.linalg import cholesky
from scipy import stats
rng = np.random.default_rng(15)

# ===== 15.5: VAR(1) MA 表示 Θ_h = A^h（用自协方差反推核对）=====
A = np.array([[0.5, 0.2], [0.1, 0.6]])
m = 2
T = 500000
Y = np.zeros((T, m))
e = rng.standard_normal((T, m))
for t in range(1, T):
    Y[t] = A @ Y[t-1] + e[t]                                        # 模拟 VAR(1)
def acvf(Y, h):
    n = Y.shape[0]; return Y[h:].T @ Y[:n-h] / n
g0, g1 = acvf(Y, 0), acvf(Y, 1)
Theta1_mc = g1 @ np.linalg.inv(g0)                                  # Θ_1 ≈ γ(1)γ(0)^{-1}
print(f"[15.5 VAR(1) MA] Θ_1 = A =\n{np.round(A, 4)}")
print(f"  MC 反推 Θ_1 =\n{np.round(Theta1_mc, 4)}  (≈A ✓)")

# ===== 15.2: VAR(2) 伴随矩阵平稳性 =====
A1 = np.array([[0.3, 0.2], [0.2, 0.3]]); A2 = np.array([[0.4, -0.1], [-0.1, 0.4]])
C = np.block([[A1, A2], [np.eye(2), np.zeros((2, 2))]])            # 伴随矩阵 4×4
ev = np.abs(np.linalg.eigvals(C))
print(f"\n[15.2 VAR(2)] 伴随矩阵 |特征值|={np.round(ev, 4)}, max={ev.max():.4f} < 1 ⇒ 平稳 ✓")

# ===== 15.12: Cholesky 正交 IRF 的排序敏感性 =====
rho = 0.8; s1, s2 = 2.0, 1.0                                        # 不等方差!
Sig = np.array([[s1**2, rho*s1*s2], [rho*s1*s2, s2**2]])
L1 = cholesky(Sig, lower=True)                                      # 排序 (Y1,Y2): 下三角
P = np.array([[0, 1], [1, 0]])                                      # 置换矩阵
Sig_perm = P @ Sig @ P.T
L2 = cholesky(Sig_perm, lower=True)                                 # 排序 (Y2,Y1)
Theta = np.array([[1.0, 0], [1.0, 1.0]])                            # 简化 IRF
oirf_12 = Theta @ L1                                                # OIRF(Y1先)
oirf_21 = P.T @ (Theta @ L2)                                        # OIRF(Y2先, 换回原序)
print(f"\n[15.12 Cholesky 排序] σ1={s1}, σ2={s2}, ρ={rho}")
print(f"  OIRF(Y1先) =\n{np.round(oirf_12, 4)}")
print(f"  OIRF(Y2先) =\n{np.round(oirf_21, 4)}")
print(f"  不同? {not np.allclose(oirf_12, oirf_21)}  ⇒ 正交 IRF 依赖排序")

# ===== 15.8/15.19: Granger 因果 Wald 检验 =====
# 模拟: Y1 Granger-causes Y2 (Y2 方程含 Y1 滞后), 反向不含
T = 1000; Y = np.zeros((T, 2)); phi_cross = 0.3
for t in range(1, T):
    Y[t, 0] = 0.5 * Y[t-1, 0] + rng.standard_normal()              # Y1: 不含 Y2 滞后
    Y[t, 1] = 0.4 * Y[t-1, 1] + phi_cross * Y[t-1, 0] + rng.standard_normal()  # Y2: 含 Y1 滞后
# 检验 Y1→Y2: Y2 方程中 Y1 滞后系数=0?
y2 = Y[1:, 1]; X = np.c_[Y[:-1, 0], Y[:-1, 1], np.ones(T-1)]
b = np.linalg.lstsq(X, y2, rcond=None)[0]; e = y2 - X @ b
V = np.linalg.inv(X.T@X) @ ((X*e[:,None]).T @ (X*e[:,None])) @ np.linalg.inv(X.T@X)
W_fwd = float(b[0]**2 / V[0, 0])
# 反向 Y2→Y1?
y1 = Y[1:, 0]; X2 = np.c_[Y[:-1, 1], Y[:-1, 0], np.ones(T-1)]
b2 = np.linalg.lstsq(X2, y1, rcond=None)[0]; e2 = y1 - X2 @ b2
V2 = np.linalg.inv(X2.T@X2) @ ((X2*e2[:,None]).T @ (X2*e2[:,None])) @ np.linalg.inv(X2.T@X2)
W_bwd = float(b2[0]**2 / V2[0, 0])
print(f"\n[Granger 因果]")
print(f"  Y1→Y2: Wald={W_fwd:.2f}, p={1-stats.chi2.cdf(W_fwd,1):.4f} (应拒绝: Y1 Granger-causes Y2)")
print(f"  Y2→Y1: Wald={W_bwd:.2f}, p={1-stats.chi2.cdf(W_bwd,1):.4f} (应不拒绝: Y2 不 Granger-cause Y1)")
